# 02 — Optymalizacja offline trajektorii roju

**Cel.** Uruchomienie pełnego pipeline'u optymalizacji *offline* (planowanie
waypointów przed startem misji) dla wybranego algorytmu meta-heurystycznego.
Notebook prezentuje przepływ identyczny z pełnymi eksperymentami z pracy
magisterskiej (Shi et al. 2020 — MSFOA; Dehghani & Trojovsky 2023 — OOA;
Xue & Shen 2020 — SSA; Deb & Jain 2014 — NSGA-III).

**Co i jak edytować, by sprawdzać inne wyniki?**

* `OPTIMIZER_NAME` — `'msffoa'`, `'ooa'`, `'ssa'`, `'nsga-3'` (mapuje na
  plik z `configs/optimizer/<name>.yaml`).
* `ENVIRONMENT_NAME` — `'empty'`, `'forest'`, `'urban'` (mapuje na
  `configs/environment/<name>.yaml`).
* `MASTER_SEED` — wartość ziarna pierwotnego dla `SeedRegistry`.
* `FAST_PARAMS` — gdy `True` parametry są drastycznie zredukowane
  (`pop_size`, `epochs`), tak by całość zajęła < 30 s. Gdy `False`,
  używane są domyślne wartości z YAML (uwaga: pełen run trwa minuty/godziny).
* `NUMBER_OF_WAYPOINTS` — liczba punktów dense trajektorii po B-spline.

**Rygor badawczy.** Wszystkie strategie używają wspólnego ewaluatora
(`VectorizedEvaluator` z 5 obiektywami F1–F5 i 3 ograniczeniami;
`src/algorithms/abstraction/trajectory/Optimization logic.md`),
wspólnej inicjalizacji populacji (`StraightLineNoiseSampling`) i wspólnego
rejestru ziaren — tym samym warunek **ceteris paribus** jest spełniony.

In [ ]:
import prepare_notebook  # noqa: F401

In [ ]:
import os
import tempfile
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import yaml

# ---------------------------------------------------------------------------
# PARAMETRY DO EDYCJI
# ---------------------------------------------------------------------------
OPTIMIZER_NAME: str = "msffoa"        # 'msffoa' | 'ooa' | 'ssa' | 'nsga-3'
ENVIRONMENT_NAME: str = "empty"       # 'empty' | 'forest' | 'urban'
MASTER_SEED: int = 43
FAST_PARAMS: bool = True               # True → szybki dry-run (~10–30 s)
NUMBER_OF_WAYPOINTS: int = 80          # docelowa liczba próbek dense trajektorii

PROJECT_ROOT = Path(prepare_notebook.project_root)
ENV_CFG_PATH = PROJECT_ROOT / "configs" / "environment" / f"{ENVIRONMENT_NAME}.yaml"
OPT_CFG_PATH = PROJECT_ROOT / "configs" / "optimizer" / f"{OPTIMIZER_NAME}.yaml"
assert ENV_CFG_PATH.exists(), f"Brak: {ENV_CFG_PATH}"
assert OPT_CFG_PATH.exists(), f"Brak: {OPT_CFG_PATH}"

In [ ]:
# Stub `HydraConfig` — pozwala wywołać strategie poza pełnym uruchomieniem
# `python main.py`. Strategie używają `HydraConfig.get().runtime.output_dir`
# do umieszczenia `optimization_history/`; przekierowujemy ten katalog
# do tymczasowego workspace'u notebooka.
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix=f"notebook02_{OPTIMIZER_NAME}_"))
print(f"Wyjście artefaktów: {OUTPUT_DIR}")

from hydra.core.hydra_config import HydraConfig
from hydra.conf import HydraConf, RuntimeConf
from omegaconf import OmegaConf

_hydra_conf = HydraConf()
_hydra_conf.runtime = RuntimeConf(choices={}, output_dir=str(OUTPUT_DIR))
HydraConfig.instance().set_config(OmegaConf.create({"hydra": _hydra_conf}))
print("HydraConfig stub aktywny:", HydraConfig.get().runtime.output_dir)

In [ ]:
from configs.environment.strategies.placement_strategies import get_placement_strategy
from src.environments.abstraction.generate_obstacles import generate_obstacles
from src.environments.abstraction.generate_world_boundaries import generate_world_boundaries
from src.environments.obstacles.ObstacleShape import ObstacleShape
from src.utils.SeedRegistry import SeedRegistry, bootstrap_global_seed, seed_numba
from src.algorithms.abstraction.count_trajectories import count_trajectories

with ENV_CFG_PATH.open("r", encoding="utf-8") as f:
    env_cfg = yaml.safe_load(f)
with OPT_CFG_PATH.open("r", encoding="utf-8") as f:
    opt_cfg = yaml.safe_load(f)

params = env_cfg["params"]
start_positions = np.asarray(env_cfg["initial_xyzs"], dtype=np.float64)
target_positions = np.asarray(env_cfg["end_xyzs"], dtype=np.float64)
drone_swarm_size = int(params["num_drones"])

print(f"Środowisko: {env_cfg['name']} ({drone_swarm_size} dronów)")
print(f"Optymalizator (YAML target): {opt_cfg['_target_']}")

In [ ]:
# Inicjalizacja ziaren — analogicznie do `main.py::main_generate`.
seeds = SeedRegistry(master_seed=MASTER_SEED)
bootstrap_global_seed(seeds.seed("global"))
seed_numba(seeds.seed("numba"))
print("Sub-seedy:", seeds.export()["children"])

In [ ]:
# Generacja świata + przeszkód (zgodnie z `GenerationDataStrategy.prepare_data`).
world_data = generate_world_boundaries(
    width=params["track_width"], length=params["track_length"],
    height=params["track_height"], ground_height=params["ground_position"],
)

placement_strategy_name = params.get("placement_strategy")
if placement_strategy_name:
    obstacles_data = generate_obstacles(
        world_data,
        n_obstacles=int(params["obstacles_number"]),
        shape_type=ObstacleShape[params["shape_type"]],
        placement_strategy=get_placement_strategy(placement_strategy_name),
        size_params={
            "length": float(params["obstacle_length"]),
            "width": float(params["obstacle_width"]),
            "height": float(params["obstacle_height"]),
        },
        start_positions=start_positions,
        target_positions=target_positions,
        safe_radius=float(params["safe_radius"]),
        rng=seeds.rng("environment"),
    )
else:
    # 'empty' — brak przeszkód.
    from src.environments.abstraction.generate_obstacles import ObstaclesData
    obstacles_data = ObstaclesData(
        data=np.zeros((0, 6), dtype=np.float64),
        shape_type=ObstacleShape.BOX,
    )
print(f"Przeszkody: {obstacles_data.count}")

In [ ]:
# Załadowanie hiperparametrów optymalizatora z YAML i (opcjonalna) redukcja
# parametrów dla szybkiego dry-runa. Wartości produkcyjne pochodzą z
# `configs/optimizer/<name>.yaml` (kalibrowane empirycznie w pracy magisterskiej).
algorithm_params = dict(opt_cfg.get("algorithm_params", {}))
if FAST_PARAMS:
    print("[FAST] Redukcja parametrów (dry-run mode).")
    if "pop_size" in algorithm_params:
        algorithm_params["pop_size"] = 16
    if "epochs" in algorithm_params:
        algorithm_params["epochs"] = 5
    if "n_gen" in algorithm_params:
        algorithm_params["n_gen"] = 5
    if "n_inner_waypoints" in algorithm_params:
        algorithm_params["n_inner_waypoints"] = 5
print("algorithm_params:", algorithm_params)

In [ ]:
# Instancjowanie strategii z konfiguracji Hydry (`_target_` + `_partial_`).
from hydra.utils import instantiate

# `_partial_: true` zwraca obiekt `functools.partial` — wystarczy do
# przekazania do `count_trajectories` jako protokół.
counting_strategy = instantiate(OmegaConf.create(opt_cfg))

t0 = time.perf_counter()
trajectories = count_trajectories(
    world_data=world_data,
    obstacles_data=obstacles_data,
    counting_protocol=lambda **kw: counting_strategy(seeds=seeds, **kw),
    drone_swarm_size=drone_swarm_size,
    number_of_waypoints=NUMBER_OF_WAYPOINTS,
    start_positions=start_positions,
    target_positions=target_positions,
    algorithm_params=algorithm_params,
)
elapsed = time.perf_counter() - t0
print(f"Czas optymalizacji: {elapsed:.2f} s")
print(f"Kształt wynikowego tensora: {trajectories.shape}  (N_drones, W, 3)")

In [ ]:
# Wykres 3D wynikowych trajektorii roju.
fig = plt.figure(figsize=(13, 6))
ax = fig.add_subplot(121, projection="3d")

colors = plt.cm.tab10(np.linspace(0, 1, drone_swarm_size))
for d in range(drone_swarm_size):
    xs, ys, zs = trajectories[d, :, 0], trajectories[d, :, 1], trajectories[d, :, 2]
    ax.plot(xs, ys, zs, color=colors[d], linewidth=2.0, label=f"drone {d}")
    ax.scatter(start_positions[d, 0], start_positions[d, 1], start_positions[d, 2],
               c=[colors[d]], s=50, marker="o", edgecolor="black", linewidth=0.6)
    ax.scatter(target_positions[d, 0], target_positions[d, 1], target_positions[d, 2],
               c=[colors[d]], s=80, marker="*", edgecolor="black", linewidth=0.6)

# Przeszkody — schematycznie jako punkty środkowe (pełny rendering w notebooku 01).
if obstacles_data.count > 0:
    ax.scatter(obstacles_data.data[:, 0], obstacles_data.data[:, 1], obstacles_data.data[:, 2],
               c="red", s=25, alpha=0.6, label="przeszkody")

ax.set_xlabel("X [m]"); ax.set_ylabel("Y [m]"); ax.set_zlabel("Z [m]")
ax.set_title(f"Trajektorie wygenerowane przez {OPTIMIZER_NAME.upper()} "
             f"w środowisku {env_cfg['name']}")
ax.legend(loc="upper right", fontsize=8)

# Drugi panel — rzut XY (widok z góry, bez deformacji 3D).
ax2 = fig.add_subplot(122)
for d in range(drone_swarm_size):
    ax2.plot(trajectories[d, :, 1], trajectories[d, :, 0], color=colors[d], linewidth=1.8)
ax2.scatter(start_positions[:, 1], start_positions[:, 0], c="#ffd54f",
            s=70, edgecolor="black", label="start", zorder=5)
ax2.scatter(target_positions[:, 1], target_positions[:, 0], c="#ef5350",
            s=100, marker="*", edgecolor="black", label="cel", zorder=5)
if obstacles_data.count > 0:
    ax2.scatter(obstacles_data.data[:, 1], obstacles_data.data[:, 0],
                c="red", s=18, alpha=0.55, label="przeszkoda")
ax2.set_xlabel("Y [m]  (kierunek misji)")
ax2.set_ylabel("X [m]  (szerokość korytarza)")
ax2.set_aspect("equal", adjustable="datalim")
ax2.grid(True, alpha=0.3, linestyle=":")
ax2.legend(fontsize=8)
ax2.set_title("Widok z góry (XY)")

plt.tight_layout()
plt.show()

In [ ]:
# Krzywa zbieżności (best-so-far per generacja) — odczyt z `optimization_history.h5`.
# `OptimizationHistoryWriter` flushuje chunk per `flush_threshold` generacji; przy
# zredukowanych parametrach (dry-run) konwergencja jest krótka.
import h5py

hist_path = OUTPUT_DIR / "optimization_history" / "optimization_history.h5"
if not hist_path.exists():
    print("[INFO] Brak pliku h5 — najprawdopodobniej żaden chunk nie został zflushowany.")
else:
    with h5py.File(hist_path, "r") as f:
        objectives = f["objectives_matrix"][...]
        feasible = f["feasible_mask"][...]
    print(f"Historia: {objectives.shape}  (n_gen, pop, n_obj)")
    n_obj = objectives.shape[-1]

    # `best_so_far` — najmniejsza suma znormalizowanych obiektywów wśród feasible.
    # Suma surowych F[:] daje sensowny proxy konwergencji (sygnaturę tę używa
    # `OptimizationHistoryWriter` w `populate_iteration_metrics.py`).
    summed = objectives.sum(axis=-1)
    summed = np.where(feasible, summed, np.inf)
    best_per_gen = summed.min(axis=1)
    best_so_far = np.minimum.accumulate(best_per_gen)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(best_so_far, marker="o", color="#1f77b4")
    ax.set_xlabel("Generacja")
    ax.set_ylabel("$\\sum F_i$ (feasible best-so-far)")
    ax.set_title(f"Krzywa zbieżności — {OPTIMIZER_NAME.upper()} / {env_cfg['name']} "
                 f"(seed={MASTER_SEED})")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Notatka metodologiczna

Domyślne wartości `pop_size`/`epochs` z `configs/optimizer/*.yaml` są
skalibrowane empirycznie (zob. `Optimization logic.md` i sekcję 2.3 pracy):

* MSFOA — `pop_size=1001, epochs=200` (≈ 200 k NFE).
* OOA   — analogicznie z mealpy (`OriginalOOA`).
* SSA   — analogicznie z mealpy (`OriginalSSA`).
* NSGA-III — `pop_size` z `n_partitions` Das-Dennis (Das & Dennis 1998).

`FAST_PARAMS=True` służy wyłącznie do walidacji pipeline'u — krzywa
konwergencji z 5 generacji nie jest reprezentatywna dla pełnego eksperymentu.
Pełne wyniki (240 runów, 30 ziaren × 2 środowiska × 4 algorytmy ×
1 avoidance) znajdują się w `appendix/A_metrics/run_metrics_subset.csv`
oraz w `results/exp_20260508_f3f718f8_bio_inspired_benchmark/`.